# ARC Prize 2026 — ARC-AGI-2 | Boyko × NVARC+ × Astra D4 Expert (V22)

## Credits & Attribution

This experimental hybrid builds on the public work of **[Konstantin Boyko — Failed in AIMO](https://www.kaggle.com/code/konstantinboyko/failed-in-aimo-on-docker-v169-v03/notebook)** and **[Koushik Rudra — Failed in AIMO](https://www.kaggle.com/code/koushikrudra/failed-in-aimo)**, incorporating multi-pass inference, cheap-first scheduling, independent seeds, and candidate pooling inspired by **[Mayank Raj — ARC-AGI-2 Triple-Pass NVARC+ SOTA](https://www.kaggle.com/code/mayankraaz/arc-agi-2-triple-pass-nvarc-sota-35-9-lb)**.

## Experimental Results

V16 preserves Boyko's Qwen3-4B BF16 model and offline environment while introducing 64 TTT augmentations, 8 DFS views, and two independent inference passes. On four local evaluation tasks, **V16 achieved 3.0/4 (4/5 test outputs)**, compared with 2.5/4 in our earlier baseline run. Importantly, **Pass A alone achieved 3.0/4**; pooling Pass A and Pass B changed one prediction but did not improve the local score. These results demonstrate a promising alternative inference configuration, although the public leaderboard impact remains unverified and the small validation sample cannot establish a consistent improvement. All original authors and their contributions are explicitly credited above.

## 📚 ARC Research — Beyond the Notebook

*Scientific research that helps us explore abstract reasoning, rule discovery, and new approaches to solving ARC challenges.*

| Research | What we can learn and why it matters |
|---|---|
| [ARC-AGI-2 — Technical Paper](https://arxiv.org/abs/2505.11831) | Abstract reasoning, compositional rules, and why the new ARC benchmark is so challenging. |
| [ARChitects — Technical Report](https://lambdalabsml.github.io/ARC2025_Solution_by_the_ARChitects/) | Depth-first search, Product-of-Experts scoring, faster candidate generation, and iterative refinement of global structures. |
| [SOAR — Evolutionary Program Synthesis](https://proceedings.mlr.press/v267/pourcel25a.html) | Generating executable Python programs, refining solutions through feedback, and learning from previous attempts. |
| [ARGA — Graph Abstractions](https://arxiv.org/abs/2210.09880) | Representing objects and spatial relationships as graphs, then searching for valid transformations under constraints. |
| [H-ARC — Human Reasoning Dataset](https://www.nature.com/articles/s41597-025-05687-1) | How humans discover abstract rules, make mistakes, and refine their reasoning when solving ARC puzzles. |

*These publications guide our ongoing research into ARC reasoning and solver design. They are provided as additional reading, not as claims that all the described methods have been implemented in this notebook.*

## Experiments

I tested the dual-pass setup on 4 local evaluation tasks (5 test outputs). Pass A solved 3/4 tasks, or 4/5 outputs. Adding Pass B changed two predictions but didn't improve the local score.

The best public LB result so far is **32.22% (v17)**. Later runs haven't beaten it.

I've also been testing a separate CPU-only symbolic solver. Adding short compositions of grid operations improved its training result from 36 to 38 correct outputs out of 1,076. There was no improvement on my small development set, so I haven't added it to the main notebook.

One thing I still need to measure properly is how many solves Pass B gains or loses compared with Pass A alone, using the same tasks and total time budget.

### 📊 Public LB | Stability Across Versions

| Metric | Original Approach | 🚀 Dual-Pass Hybrid |
|:---|:---:|:---:|
| Versions | [V10](https://www.kaggle.com/code/evgendvorkin/arc-2-2026-3-4?scriptVersionId=357676447)–[V15](https://www.kaggle.com/code/evgendvorkin/arc-2-2026-3-4?scriptVersionId=360467943) | [V17](https://www.kaggle.com/code/evgendvorkin/arc-2-2026-3-4?scriptVersionId=361249689)–[V20](https://www.kaggle.com/code/evgendvorkin/arc-2-2026-3-4?scriptVersionId=362966680) |
| Successful runs | 5 | 4 |
| Public LB range | 28.47–30.56% | **31.81–32.64%** |
| Average score | 29.22% | **32.12%** |
| Score spread | 2.09 pp | **0.83 pp** |
| Best score | 30.56% | **32.64% 🏆** |

**+2.90 pp higher average score** with a **2.5× narrower observed score range**.

*V14 failed; V16 (29.31%) is excluded from the stable-phase comparison. Observed version scores, not a controlled repeated-seed benchmark.*

## 🧩 V21 experiment — Qwen + symbolic D4 context completion

I preserved the original Qwen3-4B BF16 Dual-Pass pipeline and added a **conservative, CPU-only symbolic expert** for reconstructing rectangular occlusions from repeated grid patterns. The new mechanism was developed in a separate visual reasoning experiment and independently checked on the previously missed development task; its prediction matched that task's held-out evaluation output. The task now belongs to the development set, so this is **not an independent estimate of hidden-test performance**.

The expert runs **after** neural candidate selection, verifies the demonstrations and leave-one-out folds, and only claims a submission slot when a unique D4-based reconstruction has strongly supported context. `attempt_1` is never modified; if enabled and confident, the expert uses `attempt_2`. The notebook saves the untouched neural submission for comparison. The possibility of displacing a correct neural second attempt on unseen tasks remains a real risk; this is an experiment, not a guaranteed public LB improvement.

**Reproducibility & attribution:** This V22 derives from the original public notebooks credited above. The D4 mask-completion algorithm is an additional separately implemented symbolic expert, not a claim that the original authors wrote it. No evaluation test labels or task IDs are used in its selection logic. Set `ARC_ASTRA_PROMOTE=0` for shadow-only diagnostics; default is `1` for the experimental submission.


In [ ]:
import os, time
# V16 dual-pass budget. Hidden rerun has strict 12h limit and 15-minute write buffer.
RERUN_MODE = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))
RUN_START = time.time()
MAX_SECONDS = 12 * 3600 - 900 if RERUN_MODE else 75 * 60
# Evaluation debug run: both passes share one 75-minute budget (not 75m each).
global_end_time = RUN_START + MAX_SECONDS
print(f'V16: rerun={RERUN_MODE}; budget={MAX_SECONDS/60:.1f} min')
import subprocess
subprocess.run('uv pip uninstall tensorflow tensorflow-decision-forests keras keras-tuner keras-hub', shell=True, check=True)


In [ ]:
import os, sys, warnings; warnings.simplefilter('ignore')

is_kaggle = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "") != ""
is_rerun = os.environ.get('KAGGLE_IS_COMPETITION_RERUN', "") != "" 
is_interactive = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') == 'Interactive'  # Interactive Mode
is_batch = os.environ.get('KAGGLE_KERNEL_RUN_TYPE','') == 'Batch'
is_local = not is_kaggle
is_background = is_rerun or is_batch
is_dialog = not is_background

KAGGLE_COMPETITIONS = "/kaggle/input/competitions"
KAGGLE_DATASETS = "/kaggle/input/datasets"
KAGGLE_NOTEBOOKS = "/kaggle/input/notebooks"
KAGGLE_MODELS = "/kaggle/input/models"
KAGGLE_OUTPUT = "/kaggle/working"

def check_file(file_name):
    if file_name and not os.path.isfile(file_name):
        raise FileNotFoundError(f"File not found: {file_name}")
    return file_name

def check_dir(dir_name):
    if dir_name and not os.path.isdir(dir_name):
        raise FileNotFoundError(f"Directory not found: {dir_name}")
    return dir_name

def install_libraries(packages_dir, libraries):
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")
    subprocess.run(f"uv pip install -U --no-index --find-links {packages_dir} {libraries}", shell=True, check=True)

def install_isolation(packages_dir, libraries):
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")
    if type(libraries) is list:
        str_lib = " ".join([f"{packages_dir}/{library}" for library in libraries])
    else:
        str_lib = f"{packages_dir}/{libraries}"
    subprocess.run(f"uv pip install -U --no-index --no-deps --no-build-isolation {str_lib}", shell=True, check=True)

# # Docker-v169 unsloth==2026.7.2
# unsloth_dir = f"{KAGGLE_NOTEBOOKS}/konstantinboyko/unsloth-2026-7-2-torch-2-10-0-cu128-patched" if is_kaggle else \
#     "/kaggle/whl/unsloth/unsloth_zoo-2026.7.2-torch-2.10.0+cu128-triton-3.6.0-transformers-5.5.0-patched-d169"
# install_libraries(
#     packages_dir = unsloth_dir,
#     libraries = "unsloth==2026.7.2 unsloth_zoo==2026.7.2" \
#                 " torch==2.10.0 torchaudio==2.10.0 torchvision==0.25.0 torchao==0.17.0" \
#                 " transformers==5.5.0 accelerate==1.14.0" \
#                 " triton==3.6.0" \
#                 " numpy==2.5.1" \
#                 " xformers==0.0.35" \
#                 " peft==0.19.1 trl==0.24.0 bitsandbytes==0.49.2" \
#                 " pillow==12.3.0 datasets==4.3.0" \
#                 " wandb==0.28.0",
# )  # matplotlib==3.11.0 scikit-learn==1.9.0
# install_isolation(
#     packages_dir = f"{KAGGLE_DATASETS}/konstantinboyko/whl-flash-attn-2-8-3" if is_kaggle else "/kaggle/whl/flash-attn",
#     libraries = "flash_attn-2.8.3+cu128torch2.10-cp312-cp312-linux_x86_64.whl",
# )

# Docker v165 unsloth==2025.9.7
unsloth_dir = f"{KAGGLE_NOTEBOOKS}/konstantinboyko/unsloth-2025-9-7-torch-2-8-0-py3-12-patched" if is_kaggle else \
    "/kaggle/whl/unsloth/unsloth_zoo-2025.9.9-torch-2.8.0+cu128-triton-3.4.0-transformers-4.55.4-py312-d165-patched"
install_libraries(
    packages_dir = unsloth_dir,
    libraries = "unsloth==2025.9.7 unsloth_zoo==2025.9.9" \
                " torch==2.8.0 torchaudio==2.8.0 torchvision==0.23.0 torchao==0.14.1" \
                " transformers==4.55.4 accelerate==1.11.0" \
                " triton==3.4.0" \
                " numpy==2.2.6 matplotlib==3.10.6 scikit-learn==1.7.2" \
                " xformers==0.0.32.post2" \
                " peft==0.17.1 trl==0.22.2 bitsandbytes==0.48.2" \
                " pillow==12.0.0 datasets==3.6.0" \
                " wandb==0.20.1",
)
install_isolation(
    packages_dir = f"{KAGGLE_DATASETS}/konstantinboyko/flash-attn-2-8-2/v0.3.14" if is_kaggle \
        else "/kaggle/whl/flash-attn/github_mjun0812_flash-attention-prebuild-wheels_releases/v0.3.14",
    libraries = "flash_attn-2.8.2+cu128torch2.8-cp312-cp312-linux_x86_64.whl",
)

In [ ]:
import os, sys, time, datetime as dt, json, traceback
from pathlib import Path
from colorama import Fore, Back, Style

class mylogger(object):
    def __init__(self, fname=''):
        self.fname = fname
        self.orig_print = print
        return
    def printml(self, *args):
        #self.orig_print(args)
        toprint=''
        for arg in args:
            #if not(isinstance(arg, str) and len(arg) >= len_skip_msg and skip_msg in arg):
            self.orig_print(arg)
            try:
                toprint += json.dumps(arg, indent=2) + "\n"
            except:
                toprint += f"{arg}"
        if self.fname:
            with open(self.fname, 'at') as f:
                #f.write(f"{toprint}\n")
                f.write(f"{toprint}")
                f.flush

#begin_time = time.time()
date_time_start = dt.datetime.now()
dt_start_ymd_hms = date_time_start.strftime("%Y.%m.%d_%H-%M-%S")

start_msg = []
if is_rerun:
    is_batch = False
try:
    is_ipy_interactive = 'runtime' in get_ipython().config.IPKernelApp.connection_file
except Exception as inst:
    start_msg.append(f"Error ipy_interactive:{inst}")
    is_ipy_interactive = False
is_jupyter = "ipykernel" in globals() or 'ipykernel' in sys.modules
if not is_jupyter:
    try:
        if "IPython" in globals().get("__doc__", ""):
            is_jupyter = True
    except Exception as inst:
        start_msg.append(f"Error IPython:{inst}")
if not is_jupyter:
    try:
        from IPython import get_ipython
        if get_ipython() is not None:
            ipy = get_ipython()
            if 'IPKernelApp' in ipy.config:  # Jupyter Notebook или JupyterLab
                is_jupyter = True
            elif 'VSCODE_PID' in ipy.config:  # VS Code с Jupyter
                is_jupyter = True
            elif 'google.colab' in sys.modules:  # Google Colab (тоже Jupyter-подобное)
                is_jupyter = True
    except (ImportError, AttributeError) as inst:
        start_msg.append(f"Error get_ipython():{inst}")

# We highlight the paths so that they run correctly in Kaggle / VSCode Jupyter / VSCode Python.
file_run_path = ""
if is_jupyter:
    try:
        file_run_path = globals().get("__vsc_ipynb_file__", "")
        start_msg.append(f"file_run_path globals:{file_run_path}")
    except Exception as inst:
        start_msg.append(f"file_run_path globals:{inst}")
else:
    try:
        file_run_path = __file__
        start_msg.append(f"file_run_path:{file_run_path}")
    except Exception as inst:
        start_msg.append(f"file_run_path:{inst}")
file_run_name = Path(file_run_path).stem
version = file_run_name.split(' ')[0]
path_app = Path(file_run_path).parent
path_run = os.getcwd()
path_out = "/kaggle/working" if is_kaggle else path_app / version
Path(path_out).mkdir(parents=True, exist_ok=True)
log = mylogger(f"{path_out}/LogFile-{dt_start_ymd_hms}.txt")
print = log.printml
print(f"begin_time:{time.time()}")
for msg in start_msg:
    print(msg)
print(f"Kaggle:{is_kaggle}, Interactive Mode:{is_interactive}, Batch Mode:{is_batch}, Rerun:{is_rerun}")
print(f"is_jupyter:{is_jupyter}, is_ipy_interactive:{is_ipy_interactive}")
print(f"path_app: {path_app}")
print(f"path_run: {path_run}")
print(f"path_out: {path_out}")

def conv_to_time_str(time_f):
    return time.strftime('%H:%M:%S', time.localtime(time_f))

def conv_to_short_time_str(time_f):
    str_time = conv_to_time_str(time_f)
    parts = str_time.split(':')
    if parts[0] != '00':
        return str_time
    return parts[1] + ':' + parts[2]

def conv_to_long_time_str(time_f):
    return time.strftime('%H:%M', time.localtime(time_f))

def save_as_json(file_name, data):
    with open(f"{file_name}", "w") as file_handle:
        json.dump(data, file_handle, indent=2)

def load_from_json(file_name):
    data = None
    with open(f"{file_name}", "r") as file_handle:
        data = json.load(file_handle)
    return data

def read_jsonl_file(file_name, encoding='utf-8'):
    data = []
    with open(file_name, 'r', encoding=encoding) as f:
        data = [json.loads(line) for line in f if line.strip()]
    return data

def show_platform_info():
    if is_rerun:
        return ""
    container_name = os.environ.get('CONTAINER_NAME', '')
    container_date = os.environ.get('BUILD_DATE', '')  #.split('-')[0]
    container_vers = {'20250701':'v161'}
    if not container_name:
        container_name = container_vers.get(container_date, container_name)
    print(f"CONTAINER_NAME={container_name}, BUILD_DATE={container_date}")
    import platform
    print(f"OS Name: {platform.system()}, OS Release: {platform.release()}")
    print(f"OS Version: {platform.version()}"); print(f"Full OS Info: {platform.platform()}")
    return container_name

container_name = show_platform_info()
#SOLUTIONS_PATH = f"{path_out}/solutions"; Path(SOLUTIONS_PATH).mkdir(parents=True, exist_ok=True)
#PYTHON_RUN_PATH = f"{path_out}/python_run"; Path(PYTHON_RUN_PATH).mkdir(parents=True, exist_ok=True)
#LORA_OUT_PATH = f"{path_out}/lora"; Path(LORA_OUT_PATH).mkdir(parents=True, exist_ok=True)
VLLM_LOG_FILE = f"{path_out}/vllm-{dt_start_ymd_hms}.log"

capability_3090 = (8,6)   # RTX 3090-Ti
capability_4090 = (8,9)   # NVIDIA L4 / RTX 4090
capability_h100 = (9,0)   # NVIDIA H100 
capability_6000 = (12,0)  # NVIDIA RTX PRO 6000 Blackwell Server Edition

import torch
from torch import cuda
def show_cuda_visible_devices():
    for i in range(cuda.device_count()):
        prop = cuda.get_device_properties(i)
        capability=(prop.major, prop.minor)
        print(f"{i}. GPU name: {prop.name}, capability={capability}, "\
            f"Temperature: {cuda.temperature(i)}, Total Memory: {prop.total_memory//(1024*1024)} MB, "\
            f"Free Memory: {(prop.total_memory-cuda.device_memory_used(i))//(1024*1024)} MB")
    return capability
gpu_count = torch.cuda.device_count()
capability = show_cuda_visible_devices()

print(f"Python:{sys.version} Torch:{torch.__version__} CUDA available:{torch.cuda.is_available()} Torch CUDA:{torch.version.cuda}")
CUDA_VISIBLE_DEVICES = ','.join([str(i) for i in range(gpu_count)])
print(f"gpu_count:{gpu_count} CUDA_VISIBLE_DEVICES:{CUDA_VISIBLE_DEVICES}")
if not torch.cuda.is_available():
    raise RuntimeError("Requires a GPU.")

try:
    date_low, date_high = dt.datetime.fromisoformat("2026-04-22 00:00:00"), dt.datetime.fromisoformat("2026-04-22 20:59:59")
    is_time_window = time.time() >= date_low.timestamp() and time.time() <= date_high.timestamp()
    print(f"is_time_window:{is_time_window}")
except Exception as inst:
    start_msg.append(f"is_time_window:{inst}")
    is_time_window = False

try:
    date_low, date_high = dt.datetime.fromisoformat("2026-04-22 00:00:00"), dt.datetime.fromisoformat("2026-04-22 20:59:59")
    is_time_limit_window = time.time() >= date_low.timestamp() and time.time() <= date_high.timestamp()
    print(f"is_time_limit_window:{is_time_limit_window}")
except Exception as inst:
    start_msg.append(f"is_time_limit_window:{inst}")
    is_time_limit_window = False

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer

def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))

class QwenFormatter:
    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None

class ArcDataset:
    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)

selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]

class ArcDecoder:
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")
        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0
        correct_beam_scores = []
        for basekey, basevalues in self.decoded_results.items():
            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)
            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]
            for subkey, sample in basevalues.items():
                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])
                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"
                output_len = f"{solution.shape[0]}x{solution.shape[1]}"
                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1
        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")
        num_puzzles = len(num_tasks_per_puzzle)
        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

In [ ]:
%%writefile arc_solver.py
import os

# # For Debugging
# # Disable torch.compile autocompilation to isolate pure functions
# os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
# # Enable overwriting the compiled cache if it was created (without manually editing the cache)
# os.environ["UNSLOTH_COMPILE_OVERWRITE"] = "1"
# # # Enable compiler logging to see where exactly the code crashes
# os.environ["UNSLOTH_ENABLE_LOGGING"] = "1"

# os.environ["UNSLOTH_MOE_BACKEND"] = "unsloth_triton"  # Forced activation of the Triton MoE backend
# os.environ['UNSLOTH_RETURN_LOGITS'] = '1'
os.environ["WANDB_DISABLED"] = "true"
os.environ['WANDB_MODE'] = 'disabled'

from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter
import gc
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict
from typing import Any, Union
from transformers import DataCollatorForLanguageModeling
from contextlib import redirect_stdout, redirect_stderr
from peft import get_peft_model_state_dict, set_peft_model_state_dict
import bz2
import pickle
import logging; logging.disable(logging.WARNING)

is_kaggle = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "") != ""
is_rerun = os.environ.get('KAGGLE_IS_COMPETITION_RERUN', "") != "" 
is_interactive = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') == 'Interactive'  # Interactive Mode
is_batch = os.environ.get('KAGGLE_KERNEL_RUN_TYPE','') == 'Batch'
is_local = not is_kaggle
is_background = is_rerun or is_batch
is_dialog = not is_background

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15

class UnslothFixedTrainer(UnslothTrainer):
    # Issue https://github.com/unslothai/unsloth/issues/2435
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss
    #pass

class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch

def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:
    n = logits.size(0)
    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)
    suffixes = defaultdict(list)
    candidates = dict()
    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))
    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    while time.time() - start_time < 540 and time.time() < end_time:
        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0
        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1
        if num_alive_beams == 0:
            break
        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )
        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )
        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))
    return suffixes

@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result

@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    batch_logits = outputs.logits.float().cpu().log_softmax(-1)
    result = []
    for logits, query_tokens, answer_tokens in zip(batch_logits, batch_query_tokens, batch_answer_tokens):
        query_length = len(query_tokens)
        answer_logits = logits[query_length-1:query_length-1+len(answer_tokens)]
        answer_score = answer_logits[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
    return result

def worker(rank, queue, end_time):
    # os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    # import torch
    # torch.backends.cuda.enable_flash_sdp(True)
    # torch.backends.cuda.enable_mem_efficient_sdp(True)
    # torch.backends.cuda.enable_math_sdp(False)

    # #import torch.cuda.amp as amp
    # #amp.GradScaler = lambda **kwargs: torch.cuda.amp.GradScaler(enabled=False, **kwargs)
    # temporary_location=f"/kaggle/tmp-{rank}"
    # os.makedirs(temporary_location, exist_ok=True)

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
    train_n = int(os.getenv("ARC_N_TRAIN_AUG", "8"))
    eval_n = int(os.getenv("ARC_N_EVAL_AUG", "1"))
    lora_seed = int(os.getenv("ARC_LORA_SEED", "42"))
    train_seed = int(os.getenv("ARC_TRAIN_AUG_SEED", "1"))
    eval_seed = int(os.getenv("ARC_EVAL_AUG_SEED", "2"))
    task_cap = float(os.getenv("ARC_TASK_CAP", "950"))
    dir_outputs = os.getenv("ARC_OUT_DIR", "/kaggle/inference_outputs")
    print(f"[Rank {rank}] Boyko model; TTT={train_n}x8; DFS={eval_n}x8; lora_seed={lora_seed}; output={dir_outputs}")
    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,  # Important: Forces memory optimization
        random_state=lora_seed,
        use_rslora=True,
        loftq_config=None,
        #ensure_weight_tying=True,
        #modules_to_save=["embed_tokens", "lm_head"],
        #temporary_location=temporary_location,
    )
    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",  # "adamw_torch_fused",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=lora_seed,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",  # "steps" / "no",
        #logging_steps=1,
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )
    max_seq_length = 8192
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/kaggle/input/models/konstantinboyko/qwen3-4b-bfloat16-v02-01-01/transformers/default/1" if is_kaggle \
            else "/kaggle/input/qwen3_4b_grids15_sft139-transformers-bfloat16-v1",
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
        #dtype=torch.bfloat16,
        #attn_implementation="sdpa",
    )
    model = FastLanguageModel.get_peft_model(model, **peft_params)
    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)
    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}
    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )
    formatter = QwenFormatter(tokenizer=tokenizer)
    max_new_tokens = formatter.max_new_tokens()
    max_score = -np.log(0.2)
    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"
    arc_test_set = ArcDataset.from_file(test_path)
    # dir_outputs is configured per pass above.
    os.makedirs(dir_outputs, exist_ok=True)
    while True:
        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break
        key = queue.get()
        if key is None:
            break
        start_time = time.time()
        torch.cuda.reset_peak_memory_stats()
        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )
        model = FastLanguageModel.for_training(model)
        puzzle_ds = arc_test_set.change_keys([key])
        train_ds = puzzle_ds.augment(n=train_n, shfl_keys=True, seed=train_seed)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)
        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )
            stats = trainer.train()
            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)
            del trainer
        model = FastLanguageModel.for_inference(model)
        gc.collect()
        torch.cuda.empty_cache()
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")
        torch.cuda.reset_peak_memory_stats()
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")
        puzzle_ds_multi = puzzle_ds.split_multi_replies()
        eval_ds = puzzle_ds_multi.augment(n=eval_n, seed=eval_seed)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)
        # Geometry order in ArcDataset.augment: 0=id,1=rot90,2=rot180,3=rot270,
        # 4=transpose,5=transpose+rot90,6=transpose+rot180,7=transpose+rot270.
        # Pair geometries of identical HxW/WxH shapes for model-batched DFS.
        # This reduces exactly to Boyko's 4-per-batch grouping when eval_n=2.
        batches = []
        for side in (0, 4):
            for test_id, subkeys in test_id_to_subkeys.items():
                expected = 8 * eval_n
                if len(subkeys) != expected:
                    raise RuntimeError(f"Unexpected number of DFS views for {test_id}: {len(subkeys)} != {expected}")
                for g1, g2 in ((side, side + 2), (side + 1, side + 3)):
                    batch = subkeys[g1 * eval_n:(g1 + 1) * eval_n] + subkeys[g2 * eval_n:(g2 + 1) * eval_n]
                    if batch:
                        batches.append(batch)
        with torch.inference_mode():
            known_scores = {}
            for subkeys in batches:
                spend_time = time.time() - start_time
                if spend_time > task_cap or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break
                print(f"[Rank {rank}] decoding {subkeys}")
                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))
                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)
                for subkey_id, scored_beams in dfs_result:
                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []
                    for beam_score, tokens in scored_beams:
                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue
                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)
                        grid_id = (bk, tuple(map(tuple, solution)))
                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })
                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

In [ ]:
%%writefile starter.py
import os
import sys
import time
import json
import argparse
import traceback
import torch
import torch.multiprocessing as mp


def estimated_work(task):
    """NVARC+-inspired rough cost proxy (train token burden + test decode burden)."""
    def ntok(grid):
        return len(grid) * (len(grid[0]) + 1)
    train_tokens = sum(ntok(p["input"]) + ntok(p["output"]) for p in task["train"])
    ratios = sorted(ntok(p["output"]) / max(1, ntok(p["input"])) for p in task["train"])
    ratio = ratios[len(ratios)//2]
    test_tokens = sum(ntok(t["input"]) * (1 + ratio) for t in task["test"])
    return train_tokens * 16 + test_tokens * 8 * len(task["test"])


def local_worker(rank, queue, end_time, marker_dir):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    torch.set_default_device("cpu")
    # The distinct marker directory is cleared before EACH new pass.
    if rank > 0:
        prev = os.path.join(marker_dir, f"worker{rank-1}")
        waited = 0
        while not os.path.exists(prev) and waited < 900:
            time.sleep(5)
            waited += 5
    try:
        from arc_solver import worker
        with open(os.path.join(marker_dir, f"worker{rank}"), "w") as f:
            f.write("Ok")
        print(f"[Rank {rank}] start!")
        worker(rank, queue, end_time)
        print(f"[Rank {rank}] done!")
    except Exception as e:
        print(f"[Rank {rank}] CRASH {type(e).__name__}: {e}", flush=True)
        traceback.print_exc()


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, required=True)
    parser.add_argument("--keys-file", type=str, default="")
    parser.add_argument("--order", choices=["cheap", "sorted", "file"], default="cheap")
    args = parser.parse_args()
    rerun_mode = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
    root = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
    if not os.path.isdir(root):
        root = "/kaggle/input/arc-prize-2026-arc-agi-2"
    test_path = os.path.join(root, "arc-agi_test_challenges.json" if rerun_mode else "arc-agi_evaluation_challenges.json")
    with open(test_path) as f:
        data = json.load(f)

    if args.keys_file:
        with open(args.keys_file) as f:
            keys = [k for k in json.load(f) if k in data]
    elif rerun_mode:
        keys = sorted(data)
    else:
        debug_keys = os.getenv("ARC_DEBUG_KEYS", "0934a4d8,36a08778,981571dc,aa4ec2a5").split(",")
        keys = [k for k in debug_keys if k in data]

    if args.order == "cheap":
        keys = sorted(keys, key=lambda k: (estimated_work(data[k]), k))
    elif args.order == "sorted":
        keys = sorted(keys)

    n_gpu = min(4, torch.cuda.device_count())
    if n_gpu == 0:
        raise RuntimeError("Four L4 GPUs required for Boyko TTT; no CUDA device found")
    marker_dir = os.path.join("/kaggle/working", "v16_markers", os.environ.get("ARC_RUN_TAG", "A"))
    os.makedirs(marker_dir, exist_ok=True)
    for name in os.listdir(marker_dir):
        try:
            os.remove(os.path.join(marker_dir, name))
        except OSError:
            pass
    print(f"[starter] tasks={len(keys)} | GPUs={n_gpu} | order={args.order} | "
          f"remaining_min={(args.end_time-time.time())/60:.1f} | output={os.getenv('ARC_OUT_DIR')}",flush=True)
    queue = mp.Manager().Queue()
    for key in keys:
        queue.put(key)
    for _ in range(n_gpu):
        queue.put(None)
    mp.spawn(local_worker, args=(queue, args.end_time, marker_dir), nprocs=n_gpu)


In [ ]:
%%writefile arc2_d4_tools.py
"""Pure-Python D4 transforms and exact subgrid matches for ARC grids.

Generic utilities for a research solver. No task IDs or expected test outputs.
"""
from typing import List, Tuple

Grid = List[List[int]]


def rot90(g: Grid) -> Grid:
    """Rotate a rectangular grid counterclockwise by 90 degrees."""
    return [list(row) for row in zip(*g)][::-1]


def flip_lr(g: Grid) -> Grid:
    return [row[::-1] for row in g]


def d4(g: Grid):
    """Yield (name, transformed_grid), including all 8 transform labels."""
    a = [row[:] for row in g]
    for rot in range(4):
        yield f'R{90*rot}', a
        yield f'flipLR_R{90*rot}', flip_lr(a)
        a = rot90(a)


def occurrences(big: Grid, patch: Grid):
    """All exact matches of any D4 transform of patch in big; may include equivalent matches."""
    H, W = len(big), len(big[0])
    hits=[]
    for label, transformed in d4(patch):
        h, w = len(transformed), len(transformed[0])
        if h > H or w > W:
            continue
        for r in range(H-h+1):
            for c in range(W-w+1):
                if all(big[r+k][c:c+w] == transformed[k] for k in range(h)):
                    hits.append({'orientation':label,'row':r,'col':c,'height':h,'width':w})
    return hits


def validate_grid(g):
    if not isinstance(g,list) or not g or not isinstance(g[0],list):
        raise ValueError('Expected nonempty grid of rows')
    h,w=len(g),len(g[0])
    if not (1<=h<=30 and 1<=w<=30):
        raise ValueError(f'Grid dimensions {h}x{w} outside ARC 1..30')
    if any(len(row)!=w or any(not isinstance(v,int) or isinstance(v,bool) or not(0<=v<=9) for v in row) for row in g):
        raise ValueError('Grid must be rectangular with integer colors 0..9')
    return h,w



In [ ]:
%%writefile candidate_solver.py
"""Exact D4 context-completion expert. Standard library; no task IDs.

For each solid single-color occluder, find a completely observed donor patch
under each D4 transform and translation. Rank by the largest rectangular
known context matching exactly. Return only a uniquely best reconstructed
patch, with more observed context cells than occluded cells. The mask color
and output orientation are selected using the provided demonstrations.

Coordinates in diagnostics use half-open context bounds (top,bottom,left,right).
"""
from arc2_d4_tools import d4

def rectangles(g,color):
 pts=[(r,c) for r,row in enumerate(g) for c,v in enumerate(row) if v==color]
 if not pts:return None
 r0=min(r for r,c in pts);r1=max(r for r,c in pts);c0=min(c for r,c in pts);c1=max(c for r,c in pts)
 if len(pts)!=(r1-r0+1)*(c1-c0+1):return None
 return r0,r1,c0,c1

def best_context(g,a,dr,dc,bbox,color):
 H,W=len(g),len(g[0]);A,B=len(a),len(a[0]);r0,r1,c0,c1=bbox
 lo=max(0,-dr);hi=min(H,A-dr);left=max(0,-dc);right=min(W,B-dc)
 limits={}
 for r in range(lo,hi):
  src=a[r+dr];row=g[r]
  if not all(src[c+dc]!=color and (row[c]==color or row[c]==src[c+dc]) for c in range(c0,c1+1)):
   continue
  l=c0
  while l>left and src[l-1+dc]!=color and row[l-1]==src[l-1+dc]:l-=1
  u=c1+1
  while u<right and src[u+dc]!=color and row[u]==src[u+dc]:u+=1
  limits[r]=(l,u)
 if any(r not in limits for r in range(r0,r1+1)):return None
 top=r0
 while top>lo and top-1 in limits:top-=1
 bottom=r1+1
 while bottom<hi and bottom in limits:bottom+=1
 best=(-1,None)
 for t in range(top,r0+1):
  l=max(limits[r][0] for r in range(t,r1+1));u=min(limits[r][1] for r in range(t,r1+1))
  for b in range(r1+1,bottom+1):
   if b>r1+1:l=max(l,limits[b-1][0]);u=min(u,limits[b-1][1])
   area=(b-t)*(u-l)
   if area>best[0]:best=area,(t,b,l,u)
 return best

def search(g,color):
 box=rectangles(g,color)
 if box is None:raise ValueError('Not a rectangle')
 r0,r1,c0,c1=box;h=r1-r0+1;w=c1-c0+1; found={}
 for name,a in d4(g):
  H,W=len(a),len(a[0])
  for sr in range(H-h+1):
   for sc in range(W-w+1):
    out=[row[sc:sc+w] for row in a[sr:sr+h]]
    if any(color in row for row in out):continue
    result=best_context(g,a,sr-r0,sc-c0,box,color)
    if result is None:continue
    area,context=result;score=area-h*w
    if score<=0:continue
    key=tuple(map(tuple,out));entry={'score':score,'orientation':name,'donor_row':sr,'donor_col':sc,'context':context,'prediction':out}
    if key not in found or score>found[key]['score']:found[key]=entry
 return sorted(found.values(),key=lambda e:e['score'],reverse=True)



class Abstain(ValueError):
    """The demonstrated structural rule has no unambiguous supported prediction."""


def reconstruct(g,color):
    box=rectangles(g,color)
    if box is None: raise Abstain('Mask color does not form one solid rectangle')
    area=(box[1]-box[0]+1)*(box[3]-box[2]+1)
    ranked=search(g,color)
    if not ranked: raise Abstain('No exact context match')
    best=ranked[0]
    if best['score']<=area:
        raise Abstain('Observed exact context does not exceed unknown area')
    if len(ranked)>1 and ranked[1]['score']==best['score']:
        raise Abstain('Maximal contexts disagree about reconstruction')
    return best['prediction'],{'mask_color':color,'mask_bbox':box,
        'mask_area':area,'best_context_known_cells':best['score'],
        'runner_up_known_cells':ranked[1]['score'] if len(ranked)>1 else 0,
        'donor_orientation':best['orientation'],'donor_row':best['donor_row'],
        'donor_col':best['donor_col'],'context':best['context']}


def fit(train_pairs):
    if not train_pairs: raise Abstain('Demonstrations required')
    colors=set(v for row in train_pairs[0]['input'] for v in row)
    for p in train_pairs:
        colors.intersection_update(v for row in p['input'] for v in row)
    models=[]
    for color in sorted(colors):
        labels=None
        for p in train_pairs:
            box=rectangles(p['input'],color)
            if box is None or any(color in row for row in p['output']):
                labels=set();break
            if (box[1]-box[0]+1)*(box[3]-box[2]+1)!=len(p['output'])*len(p['output'][0]):
                labels=set();break
            try: grid,_=reconstruct(p['input'],color)
            except Abstain: labels=set();break
            matched={name for name,g in d4(grid) if g==p['output']}
            labels=matched if labels is None else labels & matched
            if not labels: break
        if labels:
            models.extend((color,label) for label in sorted(labels))
    if not models: raise Abstain('No supported model reproduces every demonstration')
    return models


def predict_candidates(train_pairs,test_input):
    outputs=[]
    for color,label in fit(train_pairs):
        try: restored,_=reconstruct(test_input,color)
        except Abstain: continue
        out=dict(d4(restored))[label]
        if out not in outputs: outputs.append(out)
    if len(outputs)!=1:
        raise Abstain('Demonstrated models do not yield one supported answer')
    return outputs


def predict(train_pairs,test_input):
    return predict_candidates(train_pairs,test_input)[0]



In [ ]:
%%writefile astra_bridge.py
"""Conservative ID-agnostic Astra expert promotion for ARC-AGI-2.

Loads only task train demonstrations and query inputs. Never reads evaluation solutions.
Baseline submission is deep copied and untouched; returns candidate overlay and diagnostics.
"""
from copy import deepcopy
import time
from candidate_solver import fit, predict_candidates, reconstruct, rectangles, Abstain
from arc2_d4_tools import validate_grid


def structural_colors(train_pairs, test_input):
    """Cheap necessary conditions, run before the expensive D4 search."""
    if len(train_pairs) < 3 or not test_input:
        return []
    common = set(v for row in train_pairs[0]['input'] for v in row)
    for p in train_pairs[1:]:
        common &= set(v for row in p['input'] for v in row)
    common &= set(v for row in test_input for v in row)
    colors = []
    for color in sorted(common):
        box = rectangles(test_input, color)
        if box is None:
            continue
        mH, mW = box[1]-box[0]+1, box[3]-box[2]+1
        if mH*mW >= len(test_input)*len(test_input[0]):
            continue
        good = True
        for pair in train_pairs:
            box2=rectangles(pair['input'],color)
            o=pair['output']
            if box2 is None or any(color in row for row in o):
                good = False;break
            h2,w2=box2[1]-box2[0]+1,box2[3]-box2[2]+1
            if (h2,w2) not in ((len(o),len(o[0])), (len(o[0]),len(o))):
                good=False;break
        if good:
            colors.append(color)
    return colors


def expert_guess(train_pairs,test_input):
    """Return (prediction, meta), or (None, meta) on any uncertainty."""
    t0=time.monotonic()
    colors=structural_colors(train_pairs,test_input)
    if not colors:
        return None,{'status':'abstain','reason':'shape/mask prefilter','sec':round(time.monotonic()-t0,3)}
    try:
        for i,p in enumerate(train_pairs):
            others=train_pairs[:i]+train_pairs[i+1:]
            outs=predict_candidates(others,p['input'])
            if p['output'] not in outs:
                return None,{'status':'abstain','reason':f'LOO fold {i} mismatch','sec':round(time.monotonic()-t0,3)}
        models=fit(train_pairs)
        if len(models)!=1:
            return None,{'status':'abstain','reason':'multiple fitted models','models':len(models),'sec':round(time.monotonic()-t0,3)}
        color,label=models[0]
        if color not in colors:
            return None,{'status':'abstain','reason':'mask prefilter contradicts fit','sec':round(time.monotonic()-t0,3)}
        restored, info=reconstruct(test_input,color)
        predicted=predict_candidates(train_pairs,test_input)
        if len(predicted)!=1:
            return None,{'status':'abstain','reason':'multiple outputs','sec':round(time.monotonic()-t0,3)}
        grid=predicted[0]
        validate_grid(grid)
        best=info['best_context_known_cells']
        next_best=info['runner_up_known_cells']
        mask_area=info['mask_area']
        # Conservative confidence gate before taking one of the two Kaggle attempts.
        if best < 50 or best < 3*mask_area or best < 4*max(1,next_best):
            return None,{'status':'abstain','reason':'context confidence gate','best':best,'runner_up':next_best,
                         'mask_area':mask_area,'sec':round(time.monotonic()-t0,3)}
        return grid,{'status':'confident','mask_area':mask_area,'best':best,'runner_up':next_best,
                     'mask_color':color,'orientation':label,'sec':round(time.monotonic()-t0,3)}
    except Exception as exc:
        return None,{'status':'abstain','reason':type(exc).__name__+': '+str(exc)[:160],
                     'sec':round(time.monotonic()-t0,3)}


def run_expert(queries, baseline_submission, promote=True, time_budget_seconds=300):
    """ID-blind; never changes attempt_1. Does not need or use ground-truth labels."""
    updated=deepcopy(baseline_submission)
    diagnostics={}
    total_start=time.monotonic()
    for task_id,task in queries.items():
        if time.monotonic()-total_start >= time_budget_seconds:
            diagnostics[task_id]={'status':'time_budget_exhausted'}
            continue
        try:
            train=task['train']
            for ti,q in enumerate(task['test']):
                if time.monotonic()-total_start >= time_budget_seconds:
                    diagnostics[f'{task_id}:{ti}']={'status':'time_budget_exhausted'}
                    continue
                grid,info=expert_guess(train,q['input'])
                info['test_index']=ti
                if grid is None:
                    if info.get('reason')!='shape/mask prefilter':
                        diagnostics[f'{task_id}:{ti}']=info
                    continue
                baseline=baseline_submission[task_id][ti]
                if grid in (baseline['attempt_1'],baseline['attempt_2']):
                    info['status']='already_in_qwen_attempts'
                elif promote:
                    updated[task_id][ti]['attempt_2']=grid
                    info['status']='promoted_to_attempt_2'
                    info['replaced_attempt_2_shape']=[len(baseline['attempt_2']),len(baseline['attempt_2'][0])]
                else:
                    info['status']='shadow_only'
                diagnostics[f'{task_id}:{ti}']=info
        except Exception as exc:
            diagnostics[task_id]={'status':'error','reason':str(exc)[:160]}
    return updated,diagnostics



In [ ]:
# Pass A: original Boyko BF16 model with NVARC+-style half-cost augmentation.
# Pass B reuses exactly the same model and environment, with independent seeds.
import os, sys, time, json, subprocess
from pathlib import Path

MY_ROOT = '/kaggle/input/competitions/arc-prize-2026-arc-agi-2'
if not os.path.isdir(MY_ROOT):
    MY_ROOT = '/kaggle/input/arc-prize-2026-arc-agi-2'
CHALLENGE_PATH = os.path.join(MY_ROOT, 'arc-agi_test_challenges.json' if RERUN_MODE else 'arc-agi_evaluation_challenges.json')
with open(CHALLENGE_PATH) as f:
    arc_challenges = json.load(f)

in_scope = (list(arc_challenges) if RERUN_MODE
            else [k for k in os.getenv('ARC_DEBUG_KEYS', '0934a4d8,36a08778,981571dc,aa4ec2a5').split(',')
                  if k in arc_challenges])

common = dict(os.environ,
    UNSLOTH_DISABLE_STATISTICS='1',
    TRITON_PTXAS_PATH='/usr/local/cuda/bin/ptxas',
    OMP_NUM_THREADS='12',
    PYTHONHASHSEED='0',
    ARC_TASK_CAP='950',
    ARC_N_TRAIN_AUG='8',
    ARC_N_EVAL_AUG='1',
)

def start_pass(name, keys, env_changes, outdir, min_remaining=0):
    remaining = global_end_time - time.time()
    if not keys or remaining < min_remaining:
        print(f'[V16 {name}] skipped: tasks={len(keys)}, remaining={remaining/60:.1f}m')
        return None
    key_file = f'/kaggle/working/v16_{name}_keys.json'
    with open(key_file, 'w') as f:
        json.dump(keys, f)
    env = dict(common, ARC_RUN_TAG=name, ARC_OUT_DIR=outdir, **{k: str(v) for k,v in env_changes.items()})
    os.makedirs(outdir,exist_ok=True)
    t = time.time()
    rc = subprocess.call(
        [sys.executable, f'{path_app}/starter.py', '--end-time',str(global_end_time), '--keys-file',key_file, '--order','cheap'],
        env=env, cwd='/kaggle/working')
    print(f'[V16 {name}] rc={rc} | {len(keys)} tasks | {(time.time()-t)/60:.1f}m | remaining={(global_end_time-time.time())/60:.1f}m')
    return rc

# Independent output stores, safe for pooled decoding.
A_DIR='/kaggle/inference_outputs'
B_DIR='/kaggle/inference_outputs_b'
passA = dict(ARC_LORA_SEED=42, ARC_TRAIN_AUG_SEED=1, ARC_EVAL_AUG_SEED=2)
passB = dict(ARC_LORA_SEED=137, ARC_TRAIN_AUG_SEED=17, ARC_EVAL_AUG_SEED=29)

# Avoid pooled results from a previous interactive run: new notebook run = clean experiment.
for clean_dir in [A_DIR, B_DIR]:
    if Path(clean_dir).is_dir():
        old_files=list(Path(clean_dir).iterdir())
        for old in old_files:
            if old.is_file(): old.unlink()
        print(f'[V16] cleared {len(old_files)} previous entries in {clean_dir}')

start_pass('A', in_scope, passA, A_DIR)

def incomplete_tasks(outdir, keys):
    """Missing all candidate files for at least one test output => incomplete task."""
    present = {p.name.split('.')[0] for p in Path(outdir).iterdir() if p.is_file()} if Path(outdir).is_dir() else set()
    return [k for k in keys if any(f'{k}_{i}' not in present for i in range(len(arc_challenges[k]['test'])))]

missing = incomplete_tasks(A_DIR, in_scope)
print(f'[V16] Pass A reached {len(in_scope)-len(missing)}/{len(in_scope)} tasks; missing={len(missing)}')
# Catch-up first; do not waste scarce time on a new seed while tasks have no answers.
if missing:
    start_pass('catchup',missing,passA,A_DIR,min_remaining=10*60)

# First pass is always preserved, second pass is opportunistic.
# On a full 12h hidden rerun, Pass B starts if at least 25 minutes remain;
# on the four-task interactive/commit run, require at least 10 minutes.
min_b = 25 * 60 if RERUN_MODE else 10 * 60
start_pass('B',in_scope,passB,B_DIR,min_remaining=min_b)
print('[V16] GPU passes ended; remaining (min):',round((global_end_time-time.time())/60,2))


In [ ]:
# Finalizer runs in a FRESH process: compatible with our proven NumPy workaround.
import subprocess, sys
finalizer_code = r'''
import os, json, numpy as np
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

root = '/kaggle/input/competitions/arc-prize-2026-arc-agi-2'
if not os.path.isdir(root):
    root = '/kaggle/input/arc-prize-2026-arc-agi-2'
rerun_mode=bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))
data = ArcDataset.from_file(os.path.join(root, 'arc-agi_test_challenges.json' if rerun_mode else 'arc-agi_evaluation_challenges.json'))
if not rerun_mode:
    data=data.load_replies(os.path.join(root, 'arc-agi_evaluation_solutions.json'))
expanded=data.split_multi_replies()

A_DIR='/kaggle/inference_outputs'
B_DIR='/kaggle/inference_outputs_b'
decA=ArcDecoder(expanded,n_guesses=2)
if os.path.isdir(A_DIR):
    decA.load_decoded_results(A_DIR)
selected_a=decA.run_selection_algo()
pooled=ArcDecoder(expanded,n_guesses=2)
if os.path.isdir(A_DIR):
    pooled.load_decoded_results(A_DIR)
numB=0
if os.path.isdir(B_DIR):
    numB=len([f for f in os.listdir(B_DIR) if os.path.isfile(os.path.join(B_DIR,f))])
    pooled.load_decoded_results(B_DIR,run_name='.second')
selected_p=pooled.run_selection_algo()

selected={}
changed=0
forced=0
for basekey in set(selected_a)|set(selected_p):
    a1=selected_a[basekey][0] if basekey in selected_a and selected_a[basekey] else None
    top=list(selected_p.get(basekey,[]))[:2]
    if a1 is not None and not any(np.array_equal(a1,g) for g in top):
        top=top[:1]+[a1] if top else [a1]
        forced+=1
    if basekey in selected_a:
        old=selected_a[basekey][:2]
        if len(old)!=len(top) or any(not np.array_equal(x,y) for x,y in zip(old,top)):
            changed+=1
    selected[basekey]=top
print(f'[V16 selector] pass-A outputs={len(selected_a)} pooled={len(selected_p)}, changed={changed}, pass-A top1 restored={forced}; B count={numB}')

submission=data.get_submission(selected)
# Safe fallback only for uncovered tasks. Never change existing neural guesses.
fallback=0
for k in data.keys:
    for i,ex in enumerate(data.queries[k]['test']):
        e=submission[k][i]
        if e['attempt_1']==[[0]]:
            e['attempt_1']=ex['input']
            fallback+=1
assert set(submission)==set(data.keys)
assert all(len(submission[k])==len(data.queries[k]['test']) for k in data.keys)
assert all(all(isinstance(e[f'attempt_{i}'],list) for i in (1,2)) for v in submission.values() for e in v)

# --- V22 symbolic expert, AFTER unchanged neural selection and fallback. ---
# Freeze the Qwen-only baseline BEFORE attempting any expert selection.
import copy
baseline_sub=copy.deepcopy(submission)
with open('/kaggle/working/qwen_baseline_submission.json','w') as f:
    json.dump(baseline_sub,f)
# Write the baseline to the actual competition path first, so any expert exception
# leaves a valid full-size submission.json behind.
with open('/kaggle/working/submission.json','w') as f:
    json.dump(baseline_sub,f)
expert_diag={}
try:
    from astra_bridge import run_expert
    promote=os.getenv('ARC_ASTRA_PROMOTE','1')=='1'
    submission,expert_diag=run_expert(data.queries,baseline_sub,promote=promote,time_budget_seconds=300)
    assert set(submission)==set(baseline_sub)
    for taskid in data.keys:
        assert len(submission[taskid])==len(baseline_sub[taskid])
        for after,before in zip(submission[taskid],baseline_sub[taskid]):
            assert after['attempt_1']==before['attempt_1'], 'Neural first attempt must be preserved'
            for name in ('attempt_1','attempt_2'):
                grid=after[name]
                assert isinstance(grid,list) and grid and isinstance(grid[0],list)
                assert 1<=len(grid)<=30 and 1<=len(grid[0])<=30
                assert all(len(row)==len(grid[0]) and all(0<=int(v)<=9 for v in row) for row in grid)
    changes=[k for k,info in expert_diag.items() if info.get('status')=='promoted_to_attempt_2']
    print(f'[V22 Astra] promote={promote} | promoted={len(changes)} | keys={changes[:20]}')
    with open('/kaggle/working/submission.json','w') as f:
        json.dump(submission,f)
except Exception as exc:
    # FULL fallback to original neural submission on any bridge/integration failure.
    submission=baseline_sub
    expert_diag['fatal_error']=type(exc).__name__+': '+str(exc)
    print('[V22 Astra] safely reverted to unchanged Qwen:',expert_diag['fatal_error'])
with open('/kaggle/working/astra_expert_diagnostics.json','w') as f:
    json.dump(expert_diag,f,indent=2)
print(f'[V22 selector] submission.json saved | task_count={len(submission)} | identity fallbacks={fallback}')

if not rerun_mode:
    print('*** LOCAL score, pass A (neural):')
    passA_sub=data.get_submission(selected_a)
    print(data.validate_submission(passA_sub))
    print('*** LOCAL score, Qwen A+B baseline:')
    print(data.validate_submission(baseline_sub))
    print('*** LOCAL score, Qwen + Astra V22:')
    print(data.validate_submission(submission))
    gained=[];lost=[]
    print('*** Per-task pass-A / Qwen baseline / V22 expert outputs:')
    for k in data.keys:
        labels=data.replies.get(k,[])
        def _match(one):
            return sum(any(np.array_equal(gt,one[k][i][f'attempt_{j}']) for j in (1,2)) for i,gt in enumerate(labels))
        a,b,v=(_match(s) for s in (passA_sub,baseline_sub,submission))
        if v>b:gained.append((k,v-b))
        if v<b:lost.append((k,b-v))
        print(k, f'{a}/{len(labels)} -> {b}/{len(labels)} -> {v}/{len(labels)}')
    print('[V22 gained/lost/net]',gained,lost,
          sum(x[1] for x in gained)-sum(x[1] for x in lost))
'''
result=subprocess.run([sys.executable,'-c',finalizer_code],cwd='/kaggle/working',check=True)
print('V16 finalizer OK:',result.returncode)


### Reading the results

In the finalizer logs compare **LOCAL score, pass A** with **LOCAL score, pooled A+B**. If pooled is better, look at `changed` and `pass-A top1 restored`. These are only four familiar local tasks, not public leaderboard evidence. Both passes use Konstantin Boyko's BF16 weights; NVARC+ ideas are the inspiration for scheduling and pooled inference. The CPU EDA v0.3 solver is deliberately excluded because challenge-only evaluation fitting found no applicable rule.
